# Stage 6 - Validation & Sanity Checks

Read-only review of the Stage 1-5 outputs. **No retraining, no rebuilding the
simulation.** Findings are written up in `report/model_validation_notes.md`.

Checks: (1) top-20 champions, (2) all group winners, (3) host nations,
(4) neutral/home encoding, (5) bracket easy-path analysis, (6) missing Elo +
imputation impact, (7) model vs Elo expectation, (8) unreasonable outputs.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import joblib

from src import config, simulation

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

feat = pd.read_parquet(config.PROCESSED_FILES["fixtures_2026_features"])
preds = pd.read_csv(config.OUTPUT_FILES["fixtures_2026_predictions"])
champions = pd.read_csv(config.OUTPUT_FILES["champion_probabilities"])
advancement = pd.read_csv(config.OUTPUT_FILES["advancement_probabilities"])
bundle = joblib.load(config.OUTPUT_FILES["best_model"])
elo = pd.read_parquet(config.INTERIM_FILES["eloratings_clean"])

## 1. Top-20 champion probabilities

In [2]:
champions.head(20).assign(champion_pct=lambda d: (d.champion_prob * 100).round(2))[
    ["team", "group", "champion_pct"]
]

,team,group,champion_pct
0,Argentina,A,6.32
1,England,H,5.28
2,France,K,5.20
3,Belgium,C,4.92
4,Canada,D,4.84
5,Brazil,E,4.80
6,Switzerland,D,4.70
7,Netherlands,L,4.58
8,Spain,F,4.53
9,Germany,I,4.36


## 2. Group-winner probabilities for all groups

In [3]:
for g in sorted(advancement.group.unique()):
    sub = advancement[advancement.group == g].sort_values("group_winner_prob", ascending=False)
    line = ", ".join(f"{r.team} {r.group_winner_prob*100:.0f}%" for _, r in sub.iterrows())
    print(f"{g}: {line}")

A: Argentina 70%, Austria 18%, Algeria 9%, Jordan 3%
B: Turkey 40%, Australia 26%, Paraguay 26%, United States 8%
C: Belgium 59%, Iran 26%, Egypt 11%, New Zealand 4%
D: Switzerland 47%, Canada 43%, Bosnia and Herzegovina 8%, Qatar 2%
E: Brazil 57%, Morocco 24%, Scotland 15%, Haiti 4%
F: Spain 57%, Uruguay 18%, Cape Verde 16%, Saudi Arabia 9%
G: Portugal 44%, Colombia 42%, Uzbekistan 7%, DR Congo 7%
H: England 57%, Croatia 34%, Panama 8%, Ghana 1%
I: Germany 47%, Ecuador 34%, Ivory Coast 17%, Curaçao 2%
J: Mexico 50%, Czech Republic 24%, South Korea 15%, South Africa 11%
K: France 57%, Norway 27%, Senegal 13%, Iraq 3%
L: Netherlands 53%, Japan 34%, Sweden 9%, Tunisia 4%


## 3. Host nations: Canada, USA, Mexico

Are the hosts given reasonable probabilities? (Watch the USA.)

In [4]:
hosts = ["Canada", "United States", "Mexico"]
advancement[advancement.team.isin(hosts)][
    ["team", "group", "group_winner_prob", "advance_top2_prob", "reach_qf", "reach_sf", "champion_prob"]
].round(4)

,team,group,group_winner_prob,advance_top2_prob,reach_qf,reach_sf,champion_prob
7,Canada,D,0.4289,0.8164,0.2958,0.1629,0.0484
10,Mexico,J,0.4983,0.7854,0.2744,0.1488,0.0418
38,United States,B,0.0785,0.2323,0.0558,0.0199,0.0019


## 4. Neutral / home-advantage encoding for 2026 fixtures

Hosts should be the only non-neutral games, and neutral games should be roughly
symmetric between home/away.

In [5]:
print("neutral value counts:", feat.neutral.value_counts().to_dict())
print("\nnon-neutral (host) fixtures:")
print(feat[feat.neutral == 0][["date", "home_team", "away_team", "neutral"]].to_string(index=False))

p = preds.merge(feat[["home_team", "away_team", "neutral"]], on=["home_team", "away_team"])
neu, host = p[p.neutral == 1], p[p.neutral == 0]
print("\nneutral games  : mean P_home=%.3f  P_away=%.3f  P_draw=%.3f" % (neu.P_home_win.mean(), neu.P_away_win.mean(), neu.P_draw.mean()))
print("host games     : mean P_home=%.3f  P_away=%.3f" % (host.P_home_win.mean(), host.P_away_win.mean()))
print("residual home bias on neutral games: %.3f" % (neu.P_home_win.mean() - neu.P_away_win.mean()))

neutral value counts: {1: 63, 0: 9}

non-neutral (host) fixtures:
      date     home_team              away_team  neutral
2026-06-11        Mexico           South Africa        0
2026-06-12        Canada Bosnia and Herzegovina        0
2026-06-12 United States               Paraguay        0
2026-06-18        Mexico            South Korea        0
2026-06-18        Canada                  Qatar        0
2026-06-19 United States              Australia        0
2026-06-24        Mexico         Czech Republic        0
2026-06-24        Canada            Switzerland        0
2026-06-25 United States                 Turkey        0

neutral games  : mean P_home=0.372  P_away=0.347  P_draw=0.281
host games     : mean P_home=0.414  P_away=0.284
residual home bias on neutral games: 0.026


## 5. Does the simplified bracket give easier paths?

Champion probability should track team strength; a low-strength team with a high
champion probability would signal a bracket artifact.

In [6]:
setup = simulation.build_setup(preds)
strength = pd.Series(setup["strength"], index=setup["teams"], name="strength")
m = advancement.merge(strength, left_on="team", right_index=True)
print("corr(strength, champion_prob):", round(np.corrcoef(m.strength, m.champion_prob)[0, 1], 3))
m = m.sort_values("strength", ascending=False).reset_index(drop=True)
m["strength_rank"] = m.index + 1
m["champ_rank"] = m.champion_prob.rank(ascending=False).astype(int)
m["rank_gap"] = m.strength_rank - m.champ_rank
print("\nlargest positive rank gaps (champ better than strength):")
m.sort_values("rank_gap", ascending=False)[["team", "group", "strength", "strength_rank", "champ_rank", "rank_gap"]].head(8)

corr(strength, champion_prob): 0.955

largest positive rank gaps (champ better than strength):


,team,group,strength,strength_rank,champ_rank,rank_gap
9,Canada,D,0.689122,10,5,5
32,Panama,H,0.403771,33,30,3
35,Bosnia and Herzegovina,D,0.380549,36,34,2
24,Czech Republic,J,0.500581,25,24,1
16,Ecuador,I,0.613780,17,16,1
13,Croatia,H,0.647075,14,13,1
38,DR Congo,G,0.334642,39,38,1
22,Paraguay,B,0.521005,23,22,1


## 6. Missing Elo in 2026 fixtures + imputation impact  (BUG)

Which teams had no Elo attached, and why? Inspecting the Elo team strings reveals
the root cause.

In [7]:
miss = sorted(set(feat[feat.home_elo.isna()].home_team) | set(feat[feat.away_elo.isna()].away_team))
print("teams missing Elo (%d):" % len(miss))
print(miss)
print("\nmedian Elo used for imputation:", bundle["impute_state"]["medians"].get("home_elo"))

# Root cause: non-breaking spaces in Elo team names.
elo_us = [t for t in elo.team.unique() if t.replace("\u00a0", " ") == "United States"]
print("\nElo entry for 'United States' exact repr:", [repr(t) for t in elo_us])
print("contains non-breaking space (\\xa0)?:", any("\u00a0" in t for t in elo_us))
print("\nchampion prob of missing-Elo teams:")
champions[champions.team.isin(miss)].round(4)

teams missing Elo (10):
['Bosnia and Herzegovina', 'Cape Verde', 'Czech Republic', 'DR Congo', 'Ivory Coast', 'New Zealand', 'Saudi Arabia', 'South Africa', 'South Korea', 'United States']

median Elo used for imputation: 1543.0

Elo entry for 'United States' exact repr: ["'United\\xa0States'"]
contains non-breaking space (\xa0)?: True

champion prob of missing-Elo teams:


,team,group,champion_prob
23,Czech Republic,J,0.0153
24,Ivory Coast,I,0.0149
27,Cape Verde,F,0.0099
31,South Korea,J,0.0075
33,Bosnia and Herzegovina,D,0.0046
35,South Africa,J,0.0044
37,DR Congo,G,0.0028
38,Saudi Arabia,F,0.0022
39,United States,B,0.0019
40,New Zealand,C,0.0015


## 7. Model predictions vs. simple Elo-difference expectation

In [8]:
d = feat.dropna(subset=["home_elo", "away_elo"]).copy()
d["elo_phome"] = 1 / (1 + 10 ** (-(d.home_elo - d.away_elo) / 400))
d = d.merge(preds, on=["home_team", "away_team"])
print("fixtures with both Elo:", len(d))
print("corr(elo_diff, P_home - P_away):", round(np.corrcoef(d.home_elo - d.away_elo, d.P_home_win - d.P_away_win)[0, 1], 3))
print("corr(elo_win_expectancy, P_home_win):", round(np.corrcoef(d.elo_phome, d.P_home_win)[0, 1], 3))
d["abs_gap"] = (d.elo_phome - d.P_home_win).abs()
print("\nbiggest model-vs-Elo gaps (model is more conservative):")
d.sort_values("abs_gap", ascending=False)[["home_team", "away_team", "home_elo", "away_elo", "elo_phome", "P_home_win", "P_away_win"]].head(8).round(3)

fixtures with both Elo: 46
corr(elo_diff, P_home - P_away): 0.977
corr(elo_win_expectancy, P_home_win): 0.966

biggest model-vs-Elo gaps (model is more conservative):


,home_team,away_team,home_elo,away_elo,elo_phome,P_home_win,P_away_win
33,Paraguay,Australia,1833.0,1774.0,0.584,0.264,0.425
35,Japan,Sweden,1878.0,1660.0,0.778,0.461,0.165
19,Netherlands,Sweden,1959.0,1660.0,0.848,0.534,0.146
37,Senegal,Iraq,1803.0,1582.0,0.781,0.485,0.203
26,Jordan,Algeria,1687.0,1726.0,0.444,0.155,0.475
30,Morocco,Haiti,1830.0,1542.0,0.840,0.557,0.138
18,Brazil,Haiti,1979.0,1542.0,0.925,0.646,0.127
9,Argentina,Algeria,2113.0,1726.0,0.903,0.631,0.080


## 8. Summary of unreasonable outputs

| Observation | Reasonable? | Cause |
|---|---|---|
| USA weakest host (~0.2% title) | **No** | **BUG** - non-breaking-space Elo join failure |
| 9 other multi-word teams under-rated | **No** | **BUG** - same root cause |
| Flat champion distribution (top ~6%) | Borderline | Conservative 3-way model + draws |
| ~2.5pp home bias on neutral games | Minor | Arbitrary home/away labelling |
| Bracket can't model fixed-draw paths | Acceptable | Documented simplification |

Full write-up: `report/model_validation_notes.md`. **The headline action item is
to strip `U+00A0` from Elo team names (and add a small alias map) in ingest/clean,
then re-run Stages 1-5.** Not applied here per the Stage-6 constraints.